In [2]:
import pandas as pd
import numpy as np
from scipy.stats import poisson
from statsmodels.formula.api import glm
from statsmodels.genmod.families import Poisson
import warnings
warnings.filterwarnings('ignore')

In [3]:
df = pd.read_csv(r"C:\Users\aryan\OneDrive\Desktop\World Cup Predictor\results_clean.csv")
df['date'] = pd.to_datetime(df['date'])
print(df.shape)
print(f"Date range: {df['date'].min().date()} to {df['date'].max().date()}")

(32101, 11)
Date range: 1990-01-12 to 2026-03-31


In [4]:
# Training data: everything BEFORE the 2022 World Cup
# 2022 WC started November 20, 2022
train_df = df[df['date'] < '2022-11-20'].copy()

# Test data: only 2022 World Cup matches
test_df = df[
    (df['date'] >= '2022-11-20') & 
    (df['tournament'] == 'FIFA World Cup')
].copy()

print(f"Training matches: {len(train_df)}")
print(f"2022 WC matches:  {len(test_df)}")
test_df[['date', 'home_team', 'away_team', 'home_score', 'away_score']].head(10)

Training matches: 28549
2022 WC matches:  64


,date,home_team,away_team,home_score,away_score
28566,2022-11-20,Qatar,Ecuador,0,2
28567,2022-11-21,Senegal,Netherlands,0,2
28568,2022-11-21,England,Iran,6,2
28569,2022-11-21,United States,Wales,1,1
28570,2022-11-22,Argentina,Saudi Arabia,1,2
28571,2022-11-22,Mexico,Poland,0,0
28572,2022-11-22,Denmark,Tunisia,0,0
28573,2022-11-22,France,Australia,4,1
28574,2022-11-23,Germany,Japan,1,2
28575,2022-11-23,Spain,Costa Rica,7,0


In [5]:
STARTING_ELO = 1500
HOME_ADVANTAGE = 100

elo_ratings = {}

def get_elo(team):
    if team not in elo_ratings:
        elo_ratings[team] = STARTING_ELO
    return elo_ratings[team]

def expected_score(rating_a, rating_b):
    return 1 / (1 + 10 ** ((rating_b - rating_a) / 400))

def update_elo(home_team, away_team, home_score, away_score, tournament_weight, neutral, date):
    home_elo = get_elo(home_team)
    away_elo = get_elo(away_team)

    if neutral:
        home_elo_adjusted = home_elo
    else:
        home_elo_adjusted = home_elo + HOME_ADVANTAGE

    home_expected = expected_score(home_elo_adjusted, away_elo)
    away_expected = expected_score(away_elo, home_elo_adjusted)

    if home_score > away_score:
        home_actual, away_actual = 1, 0
    elif home_score < away_score:
        home_actual, away_actual = 0, 1
    else:
        home_actual, away_actual = 0.5, 0.5

    year = pd.to_datetime(date).year
    if year >= 2022:
        base_k = 35
    elif year >= 2018:
        base_k = 30
    else:
        base_k = 20

    k = base_k * tournament_weight

    elo_ratings[home_team] = home_elo + k * (home_actual - home_expected)
    elo_ratings[away_team] = away_elo + k * (away_actual - away_expected)

# Process only training data
train_df = train_df.sort_values('date').reset_index(drop=True)
for _, row in train_df.iterrows():
    update_elo(
        home_team=row['home_team'],
        away_team=row['away_team'],
        home_score=row['home_score'],
        away_score=row['away_score'],
        tournament_weight=row['tournament_weight'],
        neutral=row['neutral'],
        date=row['date']
    )

elo_df = pd.DataFrame(list(elo_ratings.items()), columns=['team', 'elo'])
print(f"Teams rated: {len(elo_df)}")
print("\nTop 10 teams just before 2022 WC:")
print(elo_df.sort_values('elo', ascending=False).head(10).to_string(index=False))

Teams rated: 320

Top 10 teams just before 2022 WC:
       team         elo
  Argentina 2138.030501
     Brazil 2131.685227
Netherlands 2023.025785
      Spain 2003.360977
    Croatia 1976.562473
      Italy 1968.087616
    Uruguay 1961.998578
    Belgium 1955.373352
 Costa Rica 1939.826103
    Denmark 1929.722571


In [6]:
conf = pd.read_csv(r"C:\Users\aryan\OneDrive\Desktop\World Cup Predictor\confederation_map.csv")

# Add extra confederations for missing teams
extra_conf = {
    'Greece': 'UEFA', 'Russia': 'UEFA', 'Sweden': 'UEFA',
    'Finland': 'UEFA', 'Ireland': 'UEFA', 'Wales': 'UEFA',
    'Northern Ireland': 'UEFA', 'Iceland': 'UEFA', 'Bulgaria': 'UEFA',
    'Montenegro': 'UEFA', 'Kosovo': 'UEFA', 'North Macedonia': 'UEFA',
    'Moldova': 'UEFA', 'Belarus': 'UEFA', 'Azerbaijan': 'UEFA',
    'Armenia': 'UEFA', 'Kazakhstan': 'UEFA', 'Lithuania': 'UEFA',
    'Latvia': 'UEFA', 'Estonia': 'UEFA', 'Luxembourg': 'UEFA',
    'Malta': 'UEFA', 'Cyprus': 'UEFA', 'Zambia': 'CAF',
    'Zimbabwe': 'CAF', 'Kenya': 'CAF', 'Uganda': 'CAF',
    'China': 'AFC', 'India': 'AFC', 'Thailand': 'AFC',
    'Vietnam': 'AFC', 'Malaysia': 'AFC', 'Philippines': 'AFC',
}
extra_df = pd.DataFrame(list(extra_conf.items()), columns=['team', 'confederation'])
conf_full = pd.concat([conf, extra_df], ignore_index=True).drop_duplicates(subset='team')

# Merge Elo onto training data
train_features = train_df.copy()
train_features = train_features.merge(elo_df, left_on='home_team', right_on='team', how='left')
train_features = train_features.rename(columns={'elo': 'home_elo'}).drop(columns='team')
train_features = train_features.merge(elo_df, left_on='away_team', right_on='team', how='left')
train_features = train_features.rename(columns={'elo': 'away_elo'}).drop(columns='team')
train_features['elo_diff'] = train_features['home_elo'] - train_features['away_elo']
train_features['neutral'] = train_features['neutral'].astype(int)
train_features['fifa_rank_diff'] = 0  # not used in backtest model

train_features = train_features.dropna(subset=['home_elo', 'away_elo'])
print(f"Training features shape: {train_features.shape}")

Training features shape: (28549, 15)


In [7]:
home_model_bt = glm(
    'home_score ~ home_elo + away_elo + elo_diff + neutral + tournament_weight',
    data=train_features,
    family=Poisson()
).fit()

away_model_bt = glm(
    'away_score ~ home_elo + away_elo + elo_diff + neutral + tournament_weight',
    data=train_features,
    family=Poisson()
).fit()

print("Home model pseudo R²:", round(home_model_bt.pseudo_rsquared(), 3))
print("Away model pseudo R²:", round(away_model_bt.pseudo_rsquared(), 3))
print("Models retrained on pre-2022 data!")

Home model pseudo R²: 0.254
Away model pseudo R²: 0.209
Models retrained on pre-2022 data!


In [8]:
def predict_match_bt(home_team, away_team, neutral=True):
    home_elo = elo_ratings.get(home_team, 1500)
    away_elo = elo_ratings.get(away_team, 1500)

    match = pd.DataFrame([{
        'home_elo': home_elo,
        'away_elo': away_elo,
        'elo_diff': home_elo - away_elo,
        'neutral': int(neutral),
        'tournament_weight': 5.0,
    }])

    home_xg = home_model_bt.predict(match)[0]
    away_xg = away_model_bt.predict(match)[0]
    return home_xg, away_xg

def match_probs(home_xg, away_xg, max_goals=10):
    home_win, draw, away_win = 0, 0, 0
    for i in range(max_goals):
        for j in range(max_goals):
            p = poisson.pmf(i, home_xg) * poisson.pmf(j, away_xg)
            if i > j: home_win += p
            elif i == j: draw += p
            else: away_win += p
    return home_win, draw, away_win

# Predict all 2022 WC matches
predictions = []
for _, row in test_df.iterrows():
    home_xg, away_xg = predict_match_bt(row['home_team'], row['away_team'], neutral=True)
    hw, d, aw = match_probs(home_xg, away_xg)

    # Actual result
    if row['home_score'] > row['away_score']:
        actual = 'H'
    elif row['home_score'] < row['away_score']:
        actual = 'A'
    else:
        actual = 'D'

    predictions.append({
        'home_team': row['home_team'],
        'away_team': row['away_team'],
        'home_score': row['home_score'],
        'away_score': row['away_score'],
        'home_xg': round(home_xg, 2),
        'away_xg': round(away_xg, 2),
        'p_home_win': round(hw, 3),
        'p_draw': round(d, 3),
        'p_away_win': round(aw, 3),
        'actual': actual,
        'predicted': 'H' if hw > d and hw > aw else ('D' if d > aw else 'A')
    })

pred_df = pd.DataFrame(predictions)
print(f"Total matches predicted: {len(pred_df)}")
pred_df.head(10)

Total matches predicted: 64


,home_team,away_team,home_score,away_score,home_xg,away_xg,p_home_win,p_draw,p_away_win,actual,predicted
0,Qatar,Ecuador,0,2,1.09,1.46,0.284,0.261,0.455,A,A
1,Senegal,Netherlands,0,2,0.86,1.75,0.181,0.233,0.586,A,A
2,England,Iran,6,2,1.16,1.31,0.329,0.271,0.400,H,A
3,United States,Wales,1,1,1.56,1.02,0.500,0.253,0.247,D,H
4,Argentina,Saudi Arabia,1,2,2.11,0.64,0.716,0.184,0.100,A,H
5,Mexico,Poland,0,0,1.36,1.13,0.420,0.268,0.312,D,H
6,Denmark,Tunisia,0,0,1.69,0.88,0.565,0.240,0.195,D,H
7,France,Australia,4,1,1.47,1.03,0.473,0.262,0.265,H,H
8,Germany,Japan,1,2,1.26,1.18,0.382,0.274,0.344,A,H
9,Spain,Costa Rica,7,0,1.32,1.05,0.427,0.276,0.297,H,H


In [9]:
# 1. Simple accuracy — did we predict the right outcome?
correct = (pred_df['actual'] == pred_df['predicted']).sum()
total = len(pred_df)
accuracy = correct / total

print(f"Simple Accuracy: {correct}/{total} = {accuracy*100:.1f}%")
print()

# 2. Brier Score — measures probability calibration
# Lower is better. Random = 0.333, Perfect = 0.000
def brier_score(row):
    actual_probs = {'H': 0, 'D': 0, 'A': 0}
    actual_probs[row['actual']] = 1
    bs = ((row['p_home_win'] - actual_probs['H'])**2 +
          (row['p_draw']     - actual_probs['D'])**2 +
          (row['p_away_win'] - actual_probs['A'])**2)
    return bs

pred_df['brier_score'] = pred_df.apply(brier_score, axis=1)
mean_brier = pred_df['brier_score'].mean()
print(f"Brier Score: {mean_brier:.4f}  (lower is better, random = 0.667)")

# 3. Log Loss — punishes confident wrong predictions heavily
# Lower is better
def log_loss_row(row):
    actual_probs = {'H': 0, 'D': 0, 'A': 0}
    actual_probs[row['actual']] = 1
    eps = 1e-10
    ll = -(actual_probs['H'] * np.log(row['p_home_win'] + eps) +
           actual_probs['D'] * np.log(row['p_draw'] + eps) +
           actual_probs['A'] * np.log(row['p_away_win'] + eps))
    return ll

pred_df['log_loss'] = pred_df.apply(log_loss_row, axis=1)
mean_log_loss = pred_df['log_loss'].mean()
print(f"Log Loss:    {mean_log_loss:.4f}  (lower is better, random = 1.099)")

# 4. Result breakdown
print(f"\nPrediction breakdown:")
print(pred_df.groupby(['actual', 'predicted']).size().unstack(fill_value=0))

Simple Accuracy: 31/64 = 48.4%

Brier Score: 0.6140  (lower is better, random = 0.667)
Log Loss:    1.0296  (lower is better, random = 1.099)

Prediction breakdown:
predicted   A   H
actual           
A          13   8
D           6   9
H          10  18


In [10]:
pred_df['error'] = pred_df['actual'] != pred_df['predicted']
pred_df['surprise'] = abs(pred_df['p_home_win'] - pred_df['p_away_win'])

print("Biggest upsets we got wrong (high confidence, wrong result):")
wrong = pred_df[pred_df['error']].sort_values('brier_score', ascending=False)
print(wrong[['home_team', 'away_team', 'home_score', 'away_score', 
             'p_home_win', 'p_draw', 'p_away_win', 'actual', 'predicted']].head(10).to_string(index=False))

Biggest upsets we got wrong (high confidence, wrong result):
  home_team    away_team  home_score  away_score  p_home_win  p_draw  p_away_win actual predicted
   Cameroon       Brazil           1           0       0.084   0.159       0.757      H         A
  Argentina Saudi Arabia           1           2       0.716   0.184       0.100      A         H
South Korea        Ghana           2           3       0.646   0.208       0.146      A         H
  Argentina       France           3           3       0.604   0.234       0.162      D         H
    Belgium      Morocco           0           2       0.536   0.250       0.214      A         H
    Denmark      Tunisia           0           0       0.565   0.240       0.195      D         H
    Morocco        Spain           0           0       0.199   0.242       0.559      D         A
    Morocco      Croatia           0           0       0.220   0.249       0.531      D         A
   Cameroon       Serbia           3           3       0.

In [11]:
pred_df.to_csv(r"C:\Users\aryan\OneDrive\Desktop\World Cup Predictor\backtest_results.csv", index=False)

print("=== BACKTEST SUMMARY ===")
print(f"Tournament:      2022 FIFA World Cup")
print(f"Matches:         {len(pred_df)}")
print(f"Accuracy:        {accuracy*100:.1f}%  (random = 33.3%)")
print(f"Brier Score:     {mean_brier:.4f}  (random = 0.667)")
print(f"Log Loss:        {mean_log_loss:.4f}  (random = 1.099)")
print(f"Draws predicted: 0 out of {(pred_df['actual']=='D').sum()} actual draws")
print(f"\nMain weakness: Draw prediction → fix with Dixon-Coles")
print("\nSaved!")

=== BACKTEST SUMMARY ===
Tournament:      2022 FIFA World Cup
Matches:         64
Accuracy:        48.4%  (random = 33.3%)
Brier Score:     0.6140  (random = 0.667)
Log Loss:        1.0296  (random = 1.099)
Draws predicted: 0 out of 15 actual draws

Main weakness: Draw prediction → fix with Dixon-Coles

Saved!


In [12]:
# 2022 WC had 32 teams, 8 groups of 4
wc2022_groups = {
    'A': ['Qatar', 'Ecuador', 'Senegal', 'Netherlands'],
    'B': ['England', 'Iran', 'United States', 'Wales'],
    'C': ['Argentina', 'Saudi Arabia', 'Mexico', 'Poland'],
    'D': ['France', 'Australia', 'Denmark', 'Tunisia'],
    'E': ['Spain', 'Costa Rica', 'Germany', 'Japan'],
    'F': ['Belgium', 'Canada', 'Morocco', 'Croatia'],
    'G': ['Brazil', 'Serbia', 'Switzerland', 'Cameroon'],
    'H': ['Portugal', 'Ghana', 'Uruguay', 'South Korea'],
}

# 2022 WC fixtures (group stage)
wc2022_fixtures = []
for group, teams in wc2022_groups.items():
    for i in range(len(teams)):
        for j in range(i+1, len(teams)):
            wc2022_fixtures.append({
                'home_team': teams[i],
                'away_team': teams[j],
                'neutral': True,
                'group': group
            })

fixtures_2022 = pd.DataFrame(wc2022_fixtures)
print(f"Total group stage fixtures: {len(fixtures_2022)}")
print(fixtures_2022.head(6))

Total group stage fixtures: 48
  home_team    away_team  neutral group
0     Qatar      Ecuador     True     A
1     Qatar      Senegal     True     A
2     Qatar  Netherlands     True     A
3   Ecuador      Senegal     True     A
4   Ecuador  Netherlands     True     A
5   Senegal  Netherlands     True     A


In [13]:
def simulate_match_bt(home_team, away_team, neutral=True):
    home_xg, away_xg = predict_match_bt(home_team, away_team, neutral)
    home_goals = np.random.poisson(home_xg)
    away_goals = np.random.poisson(away_xg)
    return home_goals, away_goals

def simulate_group_2022(group_name, group_teams, fixtures_df):
    table = {team: {'W': 0, 'D': 0, 'L': 0, 'GF': 0, 'GA': 0, 'Pts': 0} for team in group_teams}
    group_fixtures = fixtures_df[fixtures_df['group'] == group_name]

    for _, row in group_fixtures.iterrows():
        home = row['home_team']
        away = row['away_team']
        hg, ag = simulate_match_bt(home, away, neutral=True)

        table[home]['GF'] += hg
        table[home]['GA'] += ag
        table[away]['GF'] += ag
        table[away]['GA'] += hg

        if hg > ag:
            table[home]['W'] += 1
            table[home]['Pts'] += 3
            table[away]['L'] += 1
        elif hg < ag:
            table[away]['W'] += 1
            table[away]['Pts'] += 3
            table[home]['L'] += 1
        else:
            table[home]['D'] += 1
            table[away]['D'] += 1
            table[home]['Pts'] += 1
            table[away]['Pts'] += 1

    table_df = pd.DataFrame(table).T
    table_df['GD'] = table_df['GF'] - table_df['GA']
    table_df = table_df.sort_values(['Pts', 'GD', 'GF'], ascending=False)
    table_df.index.name = 'team'
    return table_df.reset_index()

def simulate_knockout_bt(home_team, away_team):
    home_xg, away_xg = predict_match_bt(home_team, away_team, neutral=True)
    home_goals = np.random.poisson(home_xg)
    away_goals = np.random.poisson(away_xg)
    if home_goals == away_goals:
        if np.random.random() < 0.5:
            home_goals += 1
        else:
            away_goals += 1
    return home_team if home_goals > away_goals else away_team

print("Functions ready!")

Functions ready!


In [14]:
def simulate_wc2022():
    # Simulate all groups
    all_winners = []
    all_runners_up = []
    
    for group_name, group_teams in wc2022_groups.items():
        table = simulate_group_2022(group_name, group_teams, fixtures_2022)
        all_winners.append(table.iloc[0]['team'])
        all_runners_up.append(table.iloc[1]['team'])
    
    # Round of 16 — 2022 WC bracket pairing
    r16_pairs = [
        (all_winners[0], all_runners_up[1]),   # A1 vs B2
        (all_winners[1], all_runners_up[0]),   # B1 vs A2
        (all_winners[2], all_runners_up[3]),   # C1 vs D2
        (all_winners[3], all_runners_up[2]),   # D1 vs C2
        (all_winners[4], all_runners_up[5]),   # E1 vs F2
        (all_winners[5], all_runners_up[4]),   # F1 vs E2
        (all_winners[6], all_runners_up[7]),   # G1 vs H2
        (all_winners[7], all_runners_up[6]),   # H1 vs G2
    ]
    
    r16_winners = [simulate_knockout_bt(h, a) for h, a in r16_pairs]
    
    # Quarter finals
    qf_pairs = [
        (r16_winners[0], r16_winners[1]),
        (r16_winners[2], r16_winners[3]),
        (r16_winners[4], r16_winners[5]),
        (r16_winners[6], r16_winners[7]),
    ]
    qf_winners = [simulate_knockout_bt(h, a) for h, a in qf_pairs]
    
    # Semi finals
    sf_pairs = [
        (qf_winners[0], qf_winners[1]),
        (qf_winners[2], qf_winners[3]),
    ]
    sf_winners = [simulate_knockout_bt(h, a) for h, a in sf_pairs]
    
    # Final
    winner = simulate_knockout_bt(sf_winners[0], sf_winners[1])
    
    return {
        'r16': r16_winners,
        'qf': qf_winners,
        'sf': sf_winners,
        'winner': winner
    }

# Test one simulation
result = simulate_wc2022()
print(f"Winner: {result['winner']}")
print(f"Finalists: {result['sf']}")
print(f"Semi finalists: {result['qf']}")

Winner: United States
Finalists: ['United States', 'Morocco']
Semi finalists: ['United States', 'Argentina', 'Morocco', 'South Korea']


In [15]:
from collections import defaultdict

N = 10000
stage_counts_2022 = defaultdict(lambda: defaultdict(int))

print("Running 10,000 simulations of 2022 World Cup...")

for sim in range(N):
    if sim % 100 == 0:
        print(f"  Simulation {sim}/{N}...")
    
    result = simulate_wc2022()
    
    for team in result['r16']:
        stage_counts_2022[team]['R16'] += 1
    for team in result['qf']:
        stage_counts_2022[team]['QF'] += 1
    for team in result['sf']:
        stage_counts_2022[team]['SF'] += 1
    for team in result['winner']:
        stage_counts_2022[team]['Final'] += 1
    stage_counts_2022[result['winner']]['Winner'] += 1

print("Done!")

Running 10,000 simulations of 2022 World Cup...
  Simulation 0/10000...
  Simulation 100/10000...
  Simulation 200/10000...
  Simulation 300/10000...
  Simulation 400/10000...
  Simulation 500/10000...
  Simulation 600/10000...
  Simulation 700/10000...
  Simulation 800/10000...
  Simulation 900/10000...
  Simulation 1000/10000...
  Simulation 1100/10000...
  Simulation 1200/10000...
  Simulation 1300/10000...
  Simulation 1400/10000...
  Simulation 1500/10000...
  Simulation 1600/10000...
  Simulation 1700/10000...
  Simulation 1800/10000...
  Simulation 1900/10000...
  Simulation 2000/10000...
  Simulation 2100/10000...
  Simulation 2200/10000...
  Simulation 2300/10000...
  Simulation 2400/10000...
  Simulation 2500/10000...
  Simulation 2600/10000...
  Simulation 2700/10000...
  Simulation 2800/10000...
  Simulation 2900/10000...
  Simulation 3000/10000...
  Simulation 3100/10000...
  Simulation 3200/10000...
  Simulation 3300/10000...
  Simulation 3400/10000...
  Simulation 3500/1

In [20]:
all_teams_2022 = [team for group in wc2022_groups.values() for team in group]

results_2022 = []
for team in all_teams_2022:
    results_2022.append({
        'Team': team,
        'R16 %': round(stage_counts_2022[team]['R16'] / N * 100, 1),
        'QF %': round(stage_counts_2022[team]['QF'] / N * 100, 1),
        'SF %': round(stage_counts_2022[team]['SF'] / N * 100, 1),
        'Final %': round(stage_counts_2022[team]['Final'] / N * 100, 1),
        'Winner %': round(stage_counts_2022[team]['Winner'] / N * 100, 1),
    })

results_2022_df = pd.DataFrame(results_2022)
results_2022_df = results_2022_df.sort_values('Winner %', ascending=False).reset_index(drop=True)
results_2022_df.index += 1

# Highlight actual winner Argentina
print("2022 World Cup Simulation Results (10,000 runs)")
print("★ = Actual Winner")
print()
results_2022_df[''] = results_2022_df['Team'].apply(lambda x: '★' if x == 'Argentina' else '')
print(results_2022_df.to_string())

2022 World Cup Simulation Results (10,000 runs)
★ = Actual Winner

             Team  R16 %  QF %  SF %  Final %  Winner %   
1       Argentina   66.3  49.6  34.6     34.6      21.7  ★
2          Brazil   62.0  43.8  29.4     29.4      18.5   
3     Netherlands   53.5  36.7  19.2     19.2      10.1   
4           Spain   39.1  22.4  11.2     11.2       6.0   
5         Croatia   37.0  20.5   9.8      9.8       4.9   
6         Uruguay   35.6  16.7   8.8      8.8       4.2   
7         Belgium   35.1  17.7   7.8      7.8       3.7   
8         Denmark   34.0  15.7   8.2      8.2       3.7   
9      Costa Rica   28.1  13.7   6.0      6.0       2.7   
10           Iran   33.7  16.7   7.0      7.0       2.7   
11    Switzerland   27.7  11.3   5.4      5.4       2.5   
12         France   28.9  12.1   5.9      5.9       2.4   
13       Portugal   27.0  11.5   5.3      5.3       2.2   
14        Ecuador   26.7  12.3   4.7      4.7       1.7   
15         Serbia   22.2   8.2   3.6      3.6   

In [21]:
def show_team_pathway(team):
    r16  = round(stage_counts_2022[team]['R16'] / N * 100, 1)
    qf   = round(stage_counts_2022[team]['QF'] / N * 100, 1)
    sf   = round(stage_counts_2022[team]['SF'] / N * 100, 1)
    final = round(stage_counts_2022[team]['Final'] / N * 100, 1)
    win  = round(stage_counts_2022[team]['Winner'] / N * 100, 1)

    # Find which group this team is in
    group = [g for g, teams in wc2022_groups.items() if team in teams][0]

    print(f"\n{'='*45}")
    print(f"  {team}  (Group {group})")
    print(f"{'='*45}")
    print(f"  Group Stage  →  Round of 16:  {r16}%")
    print(f"  Round of 16  →  Quarter-final: {qf}%")
    print(f"  Quarter-final →  Semi-final:  {sf}%")
    print(f"  Semi-final   →  Final:        {final}%")
    print(f"  Final        →  Winner:       {win}%")
    print(f"{'='*45}")

# Show pathways for all 32 teams sorted by win probability
all_teams_sorted = results_2022_df['Team'].tolist()
for team in all_teams_sorted:
    show_team_pathway(team)


  Argentina  (Group C)
  Group Stage  →  Round of 16:  66.3%
  Round of 16  →  Quarter-final: 49.6%
  Quarter-final →  Semi-final:  34.6%
  Semi-final   →  Final:        34.6%
  Final        →  Winner:       21.7%

  Brazil  (Group G)
  Group Stage  →  Round of 16:  62.0%
  Round of 16  →  Quarter-final: 43.8%
  Quarter-final →  Semi-final:  29.4%
  Semi-final   →  Final:        29.4%
  Final        →  Winner:       18.5%

  Netherlands  (Group A)
  Group Stage  →  Round of 16:  53.5%
  Round of 16  →  Quarter-final: 36.7%
  Quarter-final →  Semi-final:  19.2%
  Semi-final   →  Final:        19.2%
  Final        →  Winner:       10.1%

  Spain  (Group E)
  Group Stage  →  Round of 16:  39.1%
  Round of 16  →  Quarter-final: 22.4%
  Quarter-final →  Semi-final:  11.2%
  Semi-final   →  Final:        11.2%
  Final        →  Winner:       6.0%

  Croatia  (Group F)
  Group Stage  →  Round of 16:  37.0%
  Round of 16  →  Quarter-final: 20.5%
  Quarter-final →  Semi-final:  9.8%
  Semi-fin

In [15]:
from collections import defaultdict

N = 1000
stage_counts_2022 = defaultdict(lambda: defaultdict(int))

print("Running 1,000 simulations of 2022 World Cup...")

for sim in range(N):
    if sim % 100 == 0:
        print(f"  Simulation {sim}/{N}...")
    
    result = simulate_wc2022()
    
    for team in result['r16']:
        stage_counts_2022[team]['R16'] += 1
    for team in result['qf']:
        stage_counts_2022[team]['QF'] += 1
    for team in result['sf']:
        stage_counts_2022[team]['SF'] += 1
    # Both SF winners reach the final
    for team in result['sf']:
        stage_counts_2022[team]['Final'] += 1
    stage_counts_2022[result['winner']]['Winner'] += 1

print("Done!")

Running 1,000 simulations of 2022 World Cup...
  Simulation 0/1000...
  Simulation 100/1000...
  Simulation 200/1000...
  Simulation 300/1000...
  Simulation 400/1000...
  Simulation 500/1000...
  Simulation 600/1000...
  Simulation 700/1000...
  Simulation 800/1000...
  Simulation 900/1000...
Done!


In [16]:
all_teams_2022 = [team for group in wc2022_groups.values() for team in group]

results_2022 = []
for team in all_teams_2022:
    results_2022.append({
        'Team': team,
        'R16 %': round(stage_counts_2022[team]['R16'] / N * 100, 1),
        'QF %': round(stage_counts_2022[team]['QF'] / N * 100, 1),
        'SF %': round(stage_counts_2022[team]['SF'] / N * 100, 1),
        'Final %': round(stage_counts_2022[team]['Final'] / N * 100, 1),
        'Winner %': round(stage_counts_2022[team]['Winner'] / N * 100, 1),
    })

results_2022_df = pd.DataFrame(results_2022)
results_2022_df = results_2022_df.sort_values('Winner %', ascending=False).reset_index(drop=True)
results_2022_df.index += 1

# Actual 2022 results for comparison
actual_stages = {
    'Argentina': 'WINNER',
    'France': 'Final',
    'Croatia': 'SF',
    'Morocco': 'SF',
    'Netherlands': 'QF',
    'England': 'QF',
    'Brazil': 'QF',
    'Portugal': 'QF',
}

results_2022_df['Actual'] = results_2022_df['Team'].map(actual_stages).fillna('R16 or less')
print(results_2022_df.to_string())

             Team  R16 %  QF %  SF %  Final %  Winner %       Actual
1       Argentina   65.6  49.2  35.5     35.5      21.6       WINNER
2          Brazil   62.3  44.4  31.0     31.0      18.0           QF
3     Netherlands   53.8  35.1  19.3     19.3      10.6           QF
4           Spain   38.3  22.4  11.1     11.1       7.2  R16 or less
5         Croatia   35.2  20.3   9.4      9.4       4.5           SF
6         Belgium   36.4  18.7   8.0      8.0       4.3  R16 or less
7         Denmark   34.0  14.9   8.7      8.7       4.1  R16 or less
8         Uruguay   37.8  16.2   8.1      8.1       3.6  R16 or less
9     Switzerland   27.8  11.8   5.8      5.8       3.0  R16 or less
10         France   29.8  12.3   5.9      5.9       2.5        Final
11       Portugal   24.3  10.4   5.1      5.1       2.1           QF
12         Serbia   23.8   9.4   4.5      4.5       2.1  R16 or less
13     Costa Rica   31.5  14.2   5.9      5.9       2.1  R16 or less
14           Iran   32.1  16.7   5

In [17]:
# How well did we rank the actual deep runners?
deep_runs = {
    'Argentina': 6,  # winner
    'France': 5,     # finalist
    'Croatia': 4,    # SF
    'Morocco': 4,    # SF
    'Netherlands': 3,# QF
    'England': 3,    # QF
    'Brazil': 3,     # QF
    'Portugal': 3,   # QF
}

print("=== MODEL SCORING ===\n")
print(f"{'Team':<15} {'Predicted Rank':<16} {'Actual Stage':<12} {'Verdict'}")
print("-" * 55)

for team, stage_score in sorted(deep_runs.items(), key=lambda x: -x[1]):
    rank = results_2022_df[results_2022_df['Team'] == team].index[0]
    verdict = '✅' if rank <= 10 else '⚠️' if rank <= 16 else '❌'
    stage = results_2022_df[results_2022_df['Team'] == team]['Actual'].values[0]
    print(f"{team:<15} #{rank:<15} {stage:<12} {verdict}")

print(f"\nOverall: model ranked 6/8 deep runners in top 16")
print(f"Main miss: Morocco (ranked 26th, reached SF)")
print(f"Main fix needed: Draw prediction + underdog correction")

=== MODEL SCORING ===

Team            Predicted Rank   Actual Stage Verdict
-------------------------------------------------------
Argentina       #1               WINNER       ✅
France          #10              Final        ✅
Croatia         #5               SF           ✅
Morocco         #26              SF           ❌
Netherlands     #3               QF           ✅
England         #15              QF           ⚠️
Brazil          #2               QF           ✅
Portugal        #11              QF           ⚠️

Overall: model ranked 6/8 deep runners in top 16
Main miss: Morocco (ranked 26th, reached SF)
Main fix needed: Draw prediction + underdog correction
